# Case para Ecommerce
Neste case atuei da seguinte forma:
- Indentificado que existem 4 datasets sendo 3 dimensões (Clientes, Produtos e Vendedor) e 1 fato Vendas;
- O arquivo facto vendas não possui IdVenda, sendo necessário criar um identificador;
- Os arquivos dim recebidos possuem n formatos (.txt ou .csv) e n delmitadores, então pensei sendo necessário automação para carrega-los dinamicamente de acordo com o respectivo formato
- Os campos precisam seguir a forma normal para starSchema, pois os arquivos contem campos com nomenclaturas, caracteres fora da convenção.
- Carregamento em 2 schemas no Unit Catalog (ecommerce e teamVendas)
- Modelagem starSchema, e job de carregamento no Azure SQL Database

In [0]:
# Bibliotecas Spark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import current_timestamp
from pyspark.sql.window import Window


# Bibliotecas Python auxiliares
import pandas as pd
import numpy as np
import json
import unicodedata
import datetime as dt
from datetime import datetime
from datetime import datetime, date

# Controlar partições (otimização)
spark.conf.set("spark.sql.shuffle.partitions", "200")



In [0]:
base_path = "abfss://landing@ecommerceextdl.dfs.core.windows.net/"

In [0]:
%fs ls abfss://landing@ecommerceextdl.dfs.core.windows.net

# INGESTÂO DOS DADOS I
- Criado funções abaixo que validem, formato de arquivo, delimitador de arquivo e nome do arquivo. Nesse caso carregando as bases de dimensão como Clientes, Produtos e Vendedor. Solução pra n´s formatos

In [0]:
        # cria lista de dfs fora da Vendas em todas extensões
arquivos = [
    f.name 
    for f in dbutils.fs.ls("abfss://landing@ecommerceextdl.dfs.core.windows.net/")
    if f.name.lower().endswith((".csv", ".txt", ".json", ".parquet", ".delta"))
]

In [0]:
# Captura o delmitador dinamico pra cada arquivo.
def detectar_delimitador(caminho):  
    
    
    delimitadores = [",", ";", "|", "\t"]
    melhor_delim = ","
    maior_cols = 1

    for d in delimitadores:
        try:
            df_temp = (
                spark.read
                    .option("header", "true")
                    .option("inferSchema", "true")
                    .option("delimiter", d)
                    .csv(caminho)
                    .limit(1)
            )
            qtd_cols = len(df_temp.columns)

            if qtd_cols > maior_cols:
                maior_cols = qtd_cols
                melhor_delim = d

        except:
            pass

    return melhor_delim

In [0]:
# lendo arquivo e delimitador
def ler_arquivo(nome_arquivo):
    caminho = base_path + nome_arquivo
    ext = nome_arquivo.split(".")[-1].lower()
    
    if ext in ["csv", "txt"]:
        delim = detectar_delimitador(caminho)        

        return (
            spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .option("delimiter", delim)
                .csv(caminho)
        )

    elif ext == "json":
        return spark.read.json(caminho)

    elif ext == "parquet":
        return spark.read.parquet(caminho)

    elif ext == "delta":
        return spark.read.format("delta").load(caminho)

    else:
        raise ValueError(f"Extensão não suportada: {ext}")


In [0]:
# função pra iterar o arquivo e jogar pro df padrao
def registrar_dataframes(lista_arquivos):
    
    for arquivo in lista_arquivos:        
        df = ler_arquivo(arquivo)

        # extensão
        nome_base = arquivo.split(".")[0]

        # variavel é igual nome do arquivo
        nome_var = f"df_{nome_base}"

        # Global dinamico pro nome do df
        globals()[nome_var] = df
        
registrar_dataframes(arquivos)


# INGESTÂO DOS DADOS II
Ingestão dos dados de vendas dinamicamente em um unico dataframe

In [0]:

#caminho/vendas
caminho_vendas = base_path + "Vendas/"

arquivos_vendas = [
    f.path
    for f in dbutils.fs.ls(caminho_vendas)
    if f.name.lower().endswith((".csv", ".txt"))
]
#itera sobre vendas

df_vendas = None

for arquivo in arquivos_vendas:

    nome_arquivo = arquivo.split("/")[-1]
    df_temp = ler_arquivo("Vendas/" + nome_arquivo)

    # adiciona timestamp de carregamento
    df_temp = df_temp.withColumn("dtCarga", current_timestamp())

    if df_vendas is None:
        df_vendas = df_temp
    else:
        df_vendas = df_vendas.unionByName(df_temp, allowMissingColumns=True)

    del df_temp


# Tratamento DF Vendas
Criando uma chave primária atraves da função monotonically

In [0]:
from pyspark.sql.functions import row_number, monotonically_increasing_id
from pyspark.sql.window import Window

df_vendas = df_vendas.withColumn(
    "cdVenda",    
    row_number().over(Window.orderBy(monotonically_increasing_id()))
)

df_vendas.createOrReplaceTempView("vw_fato_vendas")

# Cria datasets normalizado para as dimensões
Remove espaços e normaliza nomde dos campos em (Clientes, Produtos e Vendedor) 

In [0]:
# Função auxiliar para normalizar nomes de colunas

def normalize_colname(col):
    col = col.lower().strip()
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
    col = col.replace(" ", "_").replace("-", "_")
    return col


# Função genérica para padronizar dimensões

def padronizar_dim(df, campos_texto=None):
    
    # Renomear colunas automaticamente
    for c in df.columns:
        df = df.withColumnRenamed(c, normalize_colname(c))

    # Normalizar campos de texto
    if campos_texto:
        for c in campos_texto:
            if c in df.columns:
                df = df.withColumn(c, F.initcap(F.trim(F.col(c))))

    # Normalizar email (sempre minúsculo)
    if "email" in df.columns:
        df = df.withColumn("email", F.lower(F.trim("email")))

    # Remover duplicidades pela chave primária (cd_*)
    chave = [c for c in df.columns if c.startswith("cd_")]
    if chave:
        df = df.dropDuplicates(chave)

    return df


# Trata DimCliente

dim_df_clientes = padronizar_dim(
    df_Clientes,
    campos_texto=["nome", "pais", "regiao", "cidade", "endereco"]
)


# Trata e cria DimProduto

dim_df_produto = padronizar_dim(
    df_Produto,
    campos_texto=["desc_produto"]
)


# Trata e cria DimVendedor

dim_df_vendedor = padronizar_dim(
    df_Vendedor,
    campos_texto=["nomevendedor"]
)

# Crio view dos resultados
dim_df_clientes.createOrReplaceTempView("vw_dim_clientes")
dim_df_produto.createOrReplaceTempView("vw_dim_produto")
dim_df_vendedor.createOrReplaceTempView("vw_dim_vendedor")

# Criando as tabelas no unitCatalog dos respectivos schemas
Abaixo montei as tabelas em dois schemas:
- 1. ecommerce em uma camada bonrze com todas os dataframes tratados
- 2. no catalog teamVendas ja normalizado (com relaçoes) para consumo do time de vendas criado a vendas_full

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE ecommerce.bronze.dimCliente AS
SELECT
    cd_cliente,
    nome,
    pais,
    regiao,
    cidade,
    endereco,
    email
FROM vw_dim_clientes
""")

spark.sql("""
CREATE OR REPLACE TABLE ecommerce.bronze.dimProduto AS
SELECT
    cdproduto,
    desc_produto,
    quantidade_em_estoque
FROM vw_dim_produto
""")

spark.sql("""
CREATE OR REPLACE TABLE ecommerce.bronze.dimVendedor AS
SELECT
     cdvendedor,
     nome_vendedor

FROM vw_dim_vendedor
""")

spark.sql("""
CREATE OR REPLACE TABLE ecommerce.bronze.fatoVendas AS
SELECT
    cdVenda,
    cdCliente,
    cdProduto,
    cdVendedor,
    dtVenda,
    dtCarga,    
    valor
FROM vw_fato_vendas
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE teamvendas.gold.vendas_full AS
SELECT
    -- Fato
    f.cdVenda,
    f.cdCliente,
    f.cdProduto,
    f.cdVendedor,
    f.dtVenda,
    f.dtCarga,
    f.valor,

    -- Dim Cliente
    c.nome            AS nome_cliente,
    c.pais            AS pais_cliente,
    c.regiao          AS regiao_cliente,
    c.cidade          AS cidade_cliente,
    c.endereco        AS endereco_cliente,
    c.email           AS email_cliente,

    -- Dim Produto
    p.desc_produto,
    p.quantidade_em_estoque,

    -- Dim Vendedor
    v.nome_vendedor

FROM ecommerce.bronze.fatoVendas f
LEFT JOIN ecommerce.bronze.dimCliente  c ON f.cdCliente  = c.cd_cliente
LEFT JOIN ecommerce.bronze.dimProduto  p ON f.cdProduto  = p.cdproduto
LEFT JOIN ecommerce.bronze.dimVendedor v ON f.cdVendedor = v.cdvendedor
""")


# Gravando os dados no SQL Server pra consumo do BI via JDBC
Num ambiente real utilizaria key vault para não expor login e senha no código

In [0]:
views = {
    "DimClientes": "vw_dim_clientes",
    "DimProdutos": "vw_dim_produto",
    "DimVendedores": "vw_dim_vendedor",
    "FatoVendas": "vw_fato_vendas"
}

for nome_tabela, view_name in views.items():
    df = spark.table(view_name)

    qtd = df.count()  

    (
        df.write
        .format("jdbc")
        .option("url", "jdbc:sqlserver://bdecommerce-srv.database.windows.net:1433;database=ecommerce-db")
        .option("dbtable", nome_tabela)
        .option("user", "teamvendas")
        .option("password", "1Q2w3e4r")
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .mode("overwrite")
        .save()
    )

    print(f"Tabela {nome_tabela} carregada com sucesso — {qtd} linhas enviadas.")

# Analise complementares
- 1. Remover vendedor 10 ( erro de sistema) e exportar vendas_full para blob storage (exemplo)
- 2. Criar exportação de estudo do vendedor 10

In [0]:

df = spark.table("teamvendas.gold.vendas_full")

# Remover o vendedor '10'
df_filtrado = df.filter(df['cdvendedor'] != '10')

#  Gera arquivo na pasta
data_exportacao = datetime.now().strftime("%Y-%m-%d")
caminho = f"/mnt/landing/destinovendas/{data_exportacao}"


(
    df_filtrado
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(caminho)
)

print(f"Exportação CSV concluída com sucesso em: {caminho}")

In [0]:

#  Carrega a tabela
df = spark.table("teamvendas.gold.vendas_full")

#  Filtra somente o vendedor 10
df_v10 = df.filter(df['cdvendedor'] == '10')

#  Converte garantir um unico arquivo no blob 
pdf = df_v10.toPandas()

caminho = "/dbfs/mnt/landing/estudo/10.csv"

pdf.to_csv(caminho, index=False)

print("Arquivo salvo em:", caminho)


### FIM